# Colab Quality Eval — HUSC Admissions RAG Pipeline (v4)

*6-user-story parity & governance notebook. Restart kernel → Run All to reproduce.*

In [ ]:
import os, sys, subprocess
print(sys.version)
!nvidia-smi 2>&1 | head -5
import torch
assert torch.cuda.is_available(), "GPU chưa bật. Vào Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

## Repository Bootstrap

In [ ]:
# Clone repo — replace owner/repo before running
REPO_URL = "https://github.com/<owner>/<repo>.git"
BRANCH = "main"
!git clone -b {BRANCH} {REPO_URL} 2>&1 | tail -3
%cd husc-admission-chat-enrollment/rag2025 2>/dev/null || %cd husc-admission-chat-enrollment
!pip install -r requirements.txt -q 2>&1 | tail -5

In [ ]:
import sys
sys.path.insert(0, "/content/husc-admission-chat-enrollment")
os.environ["PYTHONPATH"] = "/content/husc-admission-chat-enrollment"
from rag2025.src.main import UnifiedQueryRequest, UnifiedQueryResponse
print("Import OK")

## Configuration & Secrets

In [ ]:
import os

# Pipeline endpoint
BASE_URL = os.getenv("PIPELINE_BASE_URL", "http://127.0.0.1:8000")

# Data paths
PRIMARY = "results/test_questions.json"
FALLBACK = "backup_mail_package_2026/python_project/rag2025/results/test_questions.json"

# API keys (set in Colab Secrets)
API_KEY = os.getenv("RAMCLOUDS_API_KEY", "") or os.getenv("OPENAI_API_KEY", "")

# Embedding models under evaluation
EMBEDDING_MODELS = ["BGE", "Harrier"]

# Routes under evaluation
ROUTES = ["padded_rag", "graph_rag"]

# Seed list for reproducibility
SEEDS = [42, 43, 44]

print(f"BASE_URL={BASE_URL}")
print(f"API_KEY set: {bool(API_KEY)}")
print(f"Embeddings: {EMBEDDING_MODELS}")
print(f"Routes: {ROUTES}")
print(f"Seeds: {SEEDS}")

## Pre-flight Check (FAIL-FAST)

In [ ]:
# FAIL-FAST: verify env before running eval
errors = []

if not API_KEY:
    errors.append("RAMCLOUDS_API_KEY or OPENAI_API_KEY not set")

try:
    import requests
    r = requests.get(f"{BASE_URL}/health", timeout=5)
    if r.status_code != 200:
        errors.append(f"Pipeline health check failed: {r.status_code}")
except Exception as e:
    errors.append(f"Cannot reach pipeline at {BASE_URL}: {e}")

if errors:
    raise RuntimeError("PREFLIGHT FAILED:\n" + "\n".join(f"  - {e}" for e in errors))

print("✅ Pre-flight PASSED")

## Data Loading

In [ ]:
from notebooks.eval_core import load_test_questions
rows, used_path = load_test_questions(PRIMARY, FALLBACK)
print(f"Loaded {len(rows)} questions from {used_path}")

from collections import Counter
cats = Counter(r.get("category", "unknown") for r in rows)
print(f"Category breakdown: {dict(cats)}")

## Decision Table (PRE-LOCK — do not modify post-run)

In [ ]:
from notebooks.eval_core import get_decision_table
import json, os

decision_table = get_decision_table()

# Save decision_table BEFORE eval runs — LOCKED
os.makedirs("results/colab_eval", exist_ok=True)
with open("results/colab_eval/decision_table.json", "w", encoding="utf-8") as f:
    json.dump(decision_table, f, indent=2, ensure_ascii=False)

print("✅ Decision table LOCKED — saved to decision_table.json")
print("\nDecision Table:")
for metric, row in decision_table.items():
    print(f"  {metric}: {row['gate_type']} | NI margin: {row['ni_margin']}")

## Matrix Evaluation (US1) — 4 Combos

Run all 4 (embedding × route) combos. FAIL-FAST if any combo is missing.

In [ ]:
from notebooks.eval_core import (
    normalize_pipeline_output,
    exact_correctness, retrieval_recall_proxy, hallucination_flag,
    latency_stats, validate_matrix_completeness, GROUNDING_THRESHOLD,
)
import time, json, requests
import pandas as pd

run_manifests = []

for embedding in EMBEDDING_MODELS:
    for route in ROUTES:
        run_id = f"{embedding}_{route}_{int(time.time())}"
        print(f"\n>>> Running: {run_id}")

        combo_results = []

        for item in rows:
            q = item["question"]
            try:
                payload = {
                    "query": q,
                    "top_k": 5,
                    "force_route": route,
                    "embedding_model": embedding,
                }
                start = time.perf_counter()
                resp = requests.post(f"{BASE_URL}/v2/query", json=payload, timeout=120)
                resp.raise_for_status()
                raw = resp.json()
                elapsed = time.perf_counter() - start

                norm = normalize_pipeline_output(raw, mode="v2")
                gt = item.get("ground_truth_answer", "")
                pred = norm["answer"]
                source_ids = norm.get("source_ids", [])
                gt_chunks = item.get("ground_truth_source_chunks", [])
                groundedness = norm.get("groundedness_score", 0.0)

                score_exact = exact_correctness(pred, gt)
                recall = retrieval_recall_proxy(source_ids, gt_chunks)
                halluc = hallucination_flag(groundedness, GROUNDING_THRESHOLD)

                combo_results.append({
                    **item,
                    **norm,
                    "score_exact": score_exact,
                    "recall": recall,
                    "hallucination": halluc,
                    "latency_ms": elapsed * 1000,
                    "embedding_model": embedding,
                    "force_route": route,
                    "run_id": run_id,
                    "requested_route": route,
                    "actual_route": norm.get("route", ""),
                })
            except Exception as e:
                combo_results.append({
                    **item,
                    "answer": "",
                    "error": str(e),
                    "score_exact": 0,
                    "recall": 0,
                    "hallucination": 1,
                    "latency_ms": 0,
                    "embedding_model": embedding,
                    "force_route": route,
                    "run_id": run_id,
                    "requested_route": route,
                    "actual_route": "",
                })

        n = len(combo_results)
        acc = sum(r["score_exact"] for r in combo_results) / n
        grounded = sum(1 - r["hallucination"] for r in combo_results) / n
        hall_rate = sum(r["hallucination"] for r in combo_results) / n
        rec = sum(r["recall"] for r in combo_results) / n
        lat_stats = latency_stats([r["latency_ms"] / 1000 for r in combo_results])

        manifest = {
            "run_id": run_id,
            "embedding_model": embedding,
            "force_route": route,
            "n_queries": n,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "accuracy": acc,
            "groundedness": grounded,
            "hallucination_rate": hall_rate,
            "recall": rec,
            "latency_p95": lat_stats["p95"],
        }
        run_manifests.append(manifest)

        print(f"  accuracy={acc:.2%} groundedness={grounded:.2%} halluc={hall_rate:.2%} recall={rec:.2%} p95={lat_stats['p95']:.3f}s")

        scored_path = f"results/colab_eval/eval_scored_{run_id}.jsonl"
        with open(scored_path, "w", encoding="utf-8") as f:
            for r in combo_results:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

        pred_path = f"results/colab_eval/eval_predictions_{run_id}.jsonl"
        with open(pred_path, "w", encoding="utf-8") as f:
            for r in combo_results:
                f.write(json.dumps({
                    "id": r.get("id"),
                    "question": r["question"],
                    "answer": r.get("answer", ""),
                    "route": r.get("actual_route", ""),
                    "embedding_model": embedding,
                    "run_id": run_id,
                }, ensure_ascii=False) + "\n")

        pd.DataFrame([manifest]).to_csv(f"results/colab_eval/eval_summary_{run_id}.csv", index=False)

validate_matrix_completeness(run_manifests)
print("\n✅ Matrix complete — all 4 combos present")

## Matrix Summary Table (US1)

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(run_manifests)
cols = ["run_id", "embedding_model", "force_route", "n_queries", "accuracy", "groundedness", "hallucination_rate", "recall", "latency_p95"]
summary_df = summary_df[cols]
print(summary_df.to_string(index=False))

# Save combined summary
summary_df.to_csv("results/colab_eval/eval_matrix_summary.csv", index=False)

## Route Parity (US2) — Controlled vs Auto-route Shadow

In [ ]:
from notebooks.eval_core import compute_route_parity
import requests, json

parity_reports = {}

for route in ROUTES:
    print(f"\n>>> Parity lane: force_route={route} vs auto-route")

    controlled_results = []
    auto_results = []

    for item in rows:
        q = item["question"]
        try:
            # Controlled lane (force route)
            ctrl_payload = {"query": q, "top_k": 5, "force_route": route}
            ctrl_resp = requests.post(f"{BASE_URL}/v2/query", json=ctrl_payload, timeout=120)
            ctrl_resp.raise_for_status()
            ctrl_norm = normalize_pipeline_output(ctrl_resp.json(), mode="v2")
            controlled_results.append({"question": q, "route": ctrl_norm.get("route", "")})

            # Auto-route lane (no force_route)
            auto_payload = {"query": q, "top_k": 5}
            auto_resp = requests.post(f"{BASE_URL}/v2/query", json=auto_payload, timeout=120)
            auto_resp.raise_for_status()
            auto_norm = normalize_pipeline_output(auto_resp.json(), mode="v2")
            auto_results.append({"question": q, "route": auto_norm.get("route", "")})

        except Exception:
            controlled_results.append({"question": q, "route": ""})
            auto_results.append({"question": q, "route": ""})

    parity = compute_route_parity(controlled_results, auto_results)
    parity["route_tested"] = route

    # Per-slice mismatch check for this lane
    per_slice_pct = (parity["mismatches"] / parity["total"] * 100) if parity["total"] > 0 else 0.0
    parity["per_slice_mismatch_pct"] = per_slice_pct
    parity["per_slice_pass"] = per_slice_pct <= 2.0

    parity_reports[route] = parity

    print(
        f"  mismatch={parity['global_mismatch_pct']:.2f}% "
        f"(global_pass={parity['global_pass']}, per_slice_pass={parity['per_slice_pass']})"
    )

with open("results/colab_eval/route_parity_report.json", "w", encoding="utf-8") as f:
    json.dump(parity_reports, f, indent=2, ensure_ascii=False)

print("\n✅ Saved: route_parity_report.json")
for route, rep in parity_reports.items():
    print(f"  {route}: global={rep['global_mismatch_pct']:.2f}% | per_slice={rep['per_slice_mismatch_pct']:.2f}%")

# Hard gate checks
if not all(rep["global_pass"] for rep in parity_reports.values()):
    raise RuntimeError("ROUTE_PARITY_FAIL: global mismatch > 1.0%")
if not all(rep["per_slice_pass"] for rep in parity_reports.values()):
    raise RuntimeError("ROUTE_PARITY_FAIL: per-slice mismatch > 2.0%")
print("✅ Route parity gates passed")

## Bootstrap CI95 (US4)

In [ ]:
from notebooks.eval_core import bootstrap_ci95

ci_results = {}

for manifest in run_manifests:
    run_id = manifest["run_id"]
    scored_path = f"results/colab_eval/eval_scored_{run_id}.jsonl"
    with open(scored_path, "r", encoding="utf-8") as f:
        scored_data = [json.loads(line) for line in f]

    for metric_key, field in [("accuracy", "score_exact"), ("groundedness", "groundedness_score"), ("recall", "recall")]:
        pass  # bootstrap over hallucination treated as binary

    # Binary metrics
    acc_vals = [r["score_exact"] for r in scored_data]
    hall_vals = [r["hallucination"] for r in scored_data]
    rec_vals = [r["recall"] for r in scored_data]
    acc_ci = bootstrap_ci95(acc_vals)
    hall_ci = bootstrap_ci95(hall_vals)
    rec_ci = bootstrap_ci95(rec_vals)

    ci_results[run_id] = {
        "accuracy_ci": acc_ci,
        "hallucination_rate_ci": hall_ci,
        "recall_ci": rec_ci,
    }
    print(f"{run_id}:")
    print(f"  accuracy  CI95: [{acc_ci[0]:.3f}, {acc_ci[1]:.3f}]")
    print(f"  hallucination CI95: [{hall_ci[0]:.3f}, {hall_ci[1]:.3f}]")
    print(f"  recall    CI95: [{rec_ci[0]:.3f}, {rec_ci[1]:.3f}]")

## Non-Inferiority Tests (US4)

In [ ]:
from notebooks.eval_core import non_inferiority_test, get_decision_table

decision_table = get_decision_table()

# Pairwise comparison: BGE vs Harrier per route
ni_results = []

for route in ROUTES:
    bge = next(m for m in run_manifests if m["embedding_model"]=="BGE" and m["force_route"]==route)
    harrier = next(m for m in run_manifests if m["embedding_model"]=="Harrier" and m["force_route"]==route)
    print(f"\n{route}: BGE acc={bge['accuracy']:.3f} Harrier acc={harrier['accuracy']:.3f}")

    for metric, ctrl_val, cand_val, higher_better in [
        ("accuracy", bge["accuracy"], harrier["accuracy"], True),
        ("groundedness", bge["groundedness"], harrier["groundedness"], True),
        ("recall", bge["recall"], harrier["recall"], True),
        ("hallucination_rate", bge["hallucination_rate"], harrier["hallucination_rate"], False),
    ]:
        ni_margin = abs(decision_table[metric]["ni_margin"])
        passes, delta = non_inferiority_test(ctrl_val, cand_val, ni_margin, higher_better=higher_better)
        winner = "Harrier" if cand_val > ctrl_val else "BGE"
        ni_results.append({
            "route": route, "metric": metric, "BGE": ctrl_val,
            "Harrier": cand_val, "delta": delta, "ni_margin": ni_margin,
            "passes": passes, "winner": winner if passes else "no winner",
        })
        status = "✅ PASS" if passes else "❌ FAIL"
        print(f"  {metric}: {status} delta={delta:+.4f} (margin={ni_margin})")

ni_df = pd.DataFrame(ni_results)
print("\nNI Test Summary:")
print(ni_df[["route","metric","passes","delta","winner"]].to_string(index=False))

## Rerun Stability (US5)

Using seeds [42, 43, 44]. Minimum 3 reruns.

In [ ]:
# NOTE: Full stability requires re-running pipeline with different seeds.
# This cell documents the stability check interface.
# For Colab runs: set SEEDS in config and re-run matrix cell.

from notebooks.eval_core import rerun_stability

# Placeholder: record stability metrics for future seed-run comparison
print("Stability check: configure SEEDS and re-run matrix to collect multi-seed data.")
print(f"Seeds configured: {SEEDS}")
print("After multi-seed runs, call: rerun_stability(seed_metrics)")

## Evidence Map (US5, US7)

In [ ]:
import hashlib, subprocess, datetime

# Git SHA
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

# Dataset hash
dataset_hash = "unknown"
try:
    with open(used_path, "rb") as f:
        dataset_hash = hashlib.sha256(f.read()).hexdigest()[:16]
except Exception:
    pass

# Config snapshot
config_snapshot = {
    "embedding_models": EMBEDDING_MODELS,
    "routes": ROUTES,
    "seeds": SEEDS,
    "base_url": BASE_URL,
    "n_queries": len(rows),
}

from notebooks.eval_core import build_evidence_map

run_ids = [m["run_id"] for m in run_manifests]
emap = build_evidence_map(git_sha, dataset_hash, config_snapshot, len(rows), run_ids)

with open("results/colab_eval/evidence_map.json", "w", encoding="utf-8") as f:
    json.dump(emap, f, indent=2, ensure_ascii=False)

print("✅ Evidence map saved:")
for k, v in emap.items():
    if k != "config_snapshot":
        print(f"  {k}: {v}")

## Enriched Diagnostic Report (US6)

In [ ]:
from notebooks.eval_core import build_enriched_diagnostic_report

# Load all scored data
all_scored = []
for manifest in run_manifests:
    run_id = manifest["run_id"]
    with open(f"results/colab_eval/eval_scored_{run_id}.jsonl", "r", encoding="utf-8") as f:
        all_scored.extend([json.loads(line) for line in f])

# Build per-route breakdown
per_route = {}
for route in ROUTES:
    route_data = [r for r in all_scored if r.get("force_route") == route]
    n = len(route_data)
    if n > 0:
        route_lat_p95 = float(pd.Series([r.get("latency_ms", 0.0) for r in route_data]).quantile(0.95) / 1000.0)
        per_route[route] = {
            "accuracy": sum(r["score_exact"] for r in route_data) / n,
            "groundedness": sum(1 - r["hallucination"] for r in route_data) / n,
            "hallucination_rate": sum(r["hallucination"] for r in route_data) / n,
            "recall": sum(r["recall"] for r in route_data) / n,
            "latency_p95": route_lat_p95,
        }

# Build per-embedding breakdown
per_embedding = {}
for emb in EMBEDDING_MODELS:
    emb_data = [r for r in all_scored if r.get("embedding_model") == emb]
    n = len(emb_data)
    if n > 0:
        per_embedding[emb] = {
            "accuracy": sum(r["score_exact"] for r in emb_data) / n,
            "groundedness": sum(1 - r["hallucination"] for r in emb_data) / n,
            "hallucination_rate": sum(r["hallucination"] for r in emb_data) / n,
            "recall": sum(r["recall"] for r in emb_data) / n,
        }

# Cross-table
cross_table = []
for cat in ["simple", "multihop"]:
    for route in ROUTES:
        for emb in EMBEDDING_MODELS:
            subset = [
                r for r in all_scored
                if r.get("category") == cat and r.get("force_route") == route and r.get("embedding_model") == emb
            ]
            n = len(subset)
            if n > 0:
                cross_table.append({
                    "category": cat,
                    "route": route,
                    "embedding": emb,
                    "count": n,
                    "accuracy": sum(r["score_exact"] for r in subset) / n,
                    "recall": sum(r["recall"] for r in subset) / n,
                })

# Errors
errors = [r for r in all_scored if r.get("score_exact", 0) == 0][:10]

# Must-pass metrics gate (all runs must satisfy minima)
def _all_runs_pass(metric_key: str, threshold: float, lower_is_better: bool = False) -> bool:
    vals = [m[metric_key] for m in run_manifests]
    if lower_is_better:
        return all(v <= threshold for v in vals)
    return all(v >= threshold for v in vals)

matrix_complete = len(run_manifests) == 4
must_pass_ok = (
    _all_runs_pass("accuracy", 0.80)
    and _all_runs_pass("groundedness", 0.75)
    and _all_runs_pass("recall", 0.70)
    and _all_runs_pass("hallucination_rate", 0.25, lower_is_better=True)
)
parity_ok = all(rep.get("global_pass", False) and rep.get("per_slice_pass", False) for rep in parity_reports.values())
evidence_ok = git_sha != "unknown" and dataset_hash != "unknown"

overall_pass = matrix_complete and must_pass_ok and parity_ok and evidence_ok

gate_decision = {
    "decision": "PASS" if overall_pass else "FAIL",
    "reasons": [
        {"check": "matrix_complete(4/4)", "result": "✅ PASS" if matrix_complete else "❌ FAIL"},
        {"check": "must_pass_metrics_all_runs", "result": "✅ PASS" if must_pass_ok else "❌ FAIL"},
        {"check": "route_parity_mismatch", "result": "✅ PASS" if parity_ok else "❌ FAIL"},
        {"check": "evidence_complete", "result": "✅ PASS" if evidence_ok else "❌ FAIL"},
    ],
}

result = {
    "summary": {
        "accuracy": sum(r["score_exact"] for r in all_scored) / len(all_scored) if all_scored else 0,
        "groundedness": sum(1 - r["hallucination"] for r in all_scored) / len(all_scored) if all_scored else 0,
        "hallucination_rate": sum(r["hallucination"] for r in all_scored) / len(all_scored) if all_scored else 0,
        "recall": sum(r["recall"] for r in all_scored) / len(all_scored) if all_scored else 0,
    },
    "per_route": per_route,
    "per_embedding": per_embedding,
    "cross_table": cross_table,
    "errors": errors,
    "parity": parity_reports,
    "gate_decision": gate_decision,
}

report = build_enriched_diagnostic_report(result)

with open("results/colab_eval/diagnostic_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print(report)